# EDA 06: kiểm chứng ngưỡng stay point bằng nhãn phương tiện

**Ngưỡng đang dùng:** D = 200 m, T = 30 phút (`src/stays/detect.py`), lấy từ bài gốc của thuật toán:
Li et al. (2008), *Mining user similarity based on location history*, ACM SIGSPATIAL GIS. Thuật toán ở Figure 2 ("Algorithm for stay point detection"); tham số ở mục 5.1.2 ("Parameter Selection"): *"we set timeThreh to 30 minutes and distThreh to 200 meters"*. Bài dùng dữ liệu đời đầu của chính dự án GeoLife (65 người, 6 tháng, ghi 2 giây/điểm). Bài nêu giá trị nhưng không giải thích vì sao chọn.

**Notebook này kiểm chứng ngưỡng đó trên dữ liệu của mình.** `labels.txt` (69 user) ghi các khoảng thời gian người dùng **đang di chuyển** (walk, bus, car...). Chạy stay point chỉ trên các điểm nằm trong những khoảng đó: mọi stay tìm được đều là **stay giả** (kẹt xe, chờ đèn, chờ tàu, đi bộ loanh quanh). Ngưỡng tốt phải cho rất ít stay giả.

**Giới hạn:**
- Chỉ đo được **nhận nhầm** (di chuyển bị coi là stay). Không đo được **bỏ sót** (lần ở lại thật bị mất), vì không có nhãn "đang ở yên".
- Chỉ 69/182 user có nhãn, và họ chỉ gắn nhãn những chuyến họ chọn.
- Một phần "stay giả" có thể là chờ thật nằm trong khoảng nhãn (người dùng bấm bắt đầu nhãn trước khi lên xe).

| Mục | Câu hỏi |
|---|---|
| 14 | Dữ liệu có nhãn gồm những gì, bao nhiêu giờ mỗi phương tiện? |
| 15 | Với từng (D, T), bao nhiêu thời gian di chuyển bị nhận nhầm là stay? |
| 16 | Ở ngưỡng đã chọn, stay giả rơi vào phương tiện nào? |
| 17 | Stay giả là đứng yên, đi loanh quanh trong một khu, hay đang chạy xe? |

Các mục sau nằm ở `07_stay_anchor.ipynb` (mục 18–21, cách chọn điểm neo) và `08_gap_stay.ipynb` (mục 22–23, gap-stay).

**Cách chạy.** Cần `data/processed/clean` (chạy `clean_trajectory.py --save-clean data/processed/clean`) và `data/interim/labels`. Run All. Chỉ chạy trên các đoạn có nhãn nên nhanh (khoảng 2–5 phút). Kết quả từng user lưu ở `data/interim/eda_stays_labels/`; đổi code stay point thì xoá thư mục đó rồi chạy lại.

Code tính toán: `src/eda/stay_scan.py`.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from eda import stay_scan as ss
from eda import split

USERS_LIMIT = None     # ví dụ ["010", "085"] để chạy thử

pd.set_option("display.max_columns", 30)

users = ss.labeled_users()
if USERS_LIMIT:
    users = [u for u in users if u in set(USERS_LIMIT)]
print(f"{len(users)} user có nhãn và có dữ liệu đã lọc")

R = ss.run(users)
print(f"{len(R) // (len(ss.D_GRID) * len(ss.T_GRID_MIN)):,} đoạn có nhãn đủ 2 điểm")

47 user có nhãn và có dữ liệu đã lọc
[10/47] user 069 (23 giây)
[20/47] user 097 (30 giây)
[30/47] user 111 (30 giây)
[40/47] user 138 (43 giây)
[47/47] user 179 (50 giây)
5,943 đoạn có nhãn đủ 2 điểm


## 14. Dữ liệu có nhãn

Số đoạn và số giờ có điểm GPS của từng phương tiện (sau lọc, đã cắt ở khoảng trống > 30 phút). Phương tiện có ít giờ thì kết quả của nó ít tin cậy.

In [2]:
mot = R[(R["D"] == ss.CHOSEN[0]) & (R["T"] == ss.CHOSEN[1])]
bang14 = mot.groupby("mode").agg(số_đoạn=("hours", "size"), số_giờ=("hours", "sum"), số_user=("user", "nunique"))
bang14 = bang14.sort_values("số_giờ", ascending=False)
bang14.loc["tổng"] = [bang14["số_đoạn"].sum(), bang14["số_giờ"].sum(), mot["user"].nunique()]
display(bang14.round(1))

,số_đoạn,số_giờ,số_user
mode,,,
walk,2222.0,930.5,42.0
car,718.0,667.5,26.0
bus,1076.0,533.6,27.0
bike,981.0,436.2,19.0
train,155.0,313.7,10.0
subway,419.0,174.6,16.0
taxi,351.0,110.6,16.0
airplane,12.0,18.5,4.0
boat,6.0,4.3,3.0


## 15. Tỷ lệ nhận nhầm theo (D, T)

- **% thời gian bị nhận nhầm**: tổng thời gian nằm trong stay giả / tổng thời gian di chuyển có nhãn. Chỉ số chính.
- **stay giả / 100 giờ di chuyển**: số lần nhận nhầm.
- **% user có stay giả**: tỷ lệ user có ít nhất một stay giả.

Hàng là D, cột là T. Ô của ngưỡng đã chọn là D = 200, T = 30.

In [3]:
g = R.groupby(["D", "T"]).agg(hours=("hours", "sum"), false_hours=("false_hours", "sum"), n_false=("n_false", "sum"))
g["% thời gian bị nhận nhầm"] = 100 * g["false_hours"] / g["hours"]
g["stay giả / 100 giờ"] = 100 * g["n_false"] / g["hours"]
pu = R.groupby(["D", "T", "user"])["n_false"].sum().gt(0).groupby(["D", "T"]).mean() * 100
g["% user có stay giả"] = pu

for cot in ["% thời gian bị nhận nhầm", "stay giả / 100 giờ", "% user có stay giả"]:
    print(cot)
    display(g[cot].unstack("T").round(2).rename_axis(index="D (m)", columns="T (phút)"))

% thời gian bị nhận nhầm


T (phút),10,20,30
D (m),,,
100,15.98,9.59,6.03
200,22.78,14.74,10.08
300,27.56,18.48,13.12


stay giả / 100 giờ


T (phút),10,20,30
D (m),,,
100,50.04,17.89,8.08
200,65.42,26.04,12.78
300,76.39,31.39,16.04


% user có stay giả


T (phút),10,20,30
D (m),,,
100,77.78,68.89,55.56
200,88.89,77.78,64.44
300,91.11,80.00,66.67


## 16. Stay giả theo phương tiện (D = 200 m)

So T = 10, 20, 30 phút ở D = 200 m. Cột là % thời gian của phương tiện đó bị nhận nhầm là stay. Stay giả tập trung ở đâu cho biết kiểu dừng nào dễ bị nhầm: tàu, máy bay (chờ ở ga, sân bay), xe buýt (kẹt xe), đi bộ (loanh quanh trong khu nhỏ).

In [4]:
m = R[R["D"] == ss.CHOSEN[0]].groupby(["mode", "T"]).agg(hours=("hours", "sum"), fh=("false_hours", "sum"))
bang16 = (100 * m["fh"] / m["hours"]).unstack("T").round(2)
bang16.columns = [f"T = {c} phút (%)" for c in bang16.columns]
bang16["số giờ có nhãn"] = m["hours"].xs(ss.CHOSEN[1], level="T").round(0)
display(bang16.sort_values("số giờ có nhãn", ascending=False))

,T = 10 phút (%),T = 20 phút (%),T = 30 phút (%),số giờ có nhãn
mode,,,,
walk,52.08,37.28,27.31,931.0
car,11.19,5.76,3.38,668.0
bus,9.22,4.03,2.03,534.0
bike,17.59,9.79,5.37,436.0
train,5.48,2.63,0.91,314.0
subway,6.58,2.78,0.75,175.0
taxi,8.04,4.47,4.09,111.0
airplane,8.31,2.00,0.00,19.0
boat,19.43,9.88,0.00,4.0


## 17. Stay giả trông như thế nào (D = 200 m, T = 30 phút)

Nhãn "đang di chuyển" không có nghĩa là mọi lúc trong nhãn đều đang di chuyển: người dùng có thể gắn nhãn `walk` cho cả buổi dạo trong khuôn viên, công viên, trung tâm thương mại, hoặc quên kết thúc nhãn khi đã dừng. Mục này xem các stay giả để phân biệt:
- **đứng yên trong nhãn**: tốc độ trung vị rất thấp (< 1 km/h). Nhãn bao cả lúc dừng, thuật toán không sai.
- **đi loanh quanh trong một khu nhỏ**: tốc độ đi bộ nhưng `rad95_m` nhỏ (khoảng 100 m). Đây là "ở một nơi" theo nghĩa địa điểm (POI), dù nhãn ghi là đi bộ.
- **đang chạy xe**: tốc độ cao. Đây mới là nhận nhầm thật sự.

Lưu ý: nhiễu GPS khi đứng yên ở tần suất 1 điểm/giây có thể cho tốc độ biểu kiến 1–3 km/h, nên tốc độ thấp không phân biệt rõ đứng yên với đi rất chậm.

In [5]:
FS = ss.false_stay_details(users)
print(f"{len(FS)} stay giả, tổng {FS['stay_min'].sum() / 60:.0f} giờ")
bang17 = FS.groupby("mode").agg(số_stay=("stay_min", "size"), số_giờ=("stay_min", lambda x: x.sum() / 60),
                                tốc_độ_trung_vị_kmh=("med_kmh", "median"), quãng_đường_km=("path_km", "median"),
                                rad95_m=("rad95_m", "median"), độ_dài_nhãn_giờ=("lab_h", "median"),
                                chạm_đầu_cuối_nhãn=("at_label_edge", "mean"))
display(bang17.sort_values("số_giờ", ascending=False).round(2))

nhom = pd.cut(FS["med_kmh"], [-0.1, 1, 3, 6, np.inf], labels=["< 1 km/h", "1–3 km/h", "3–6 km/h", "> 6 km/h"])
display(FS.groupby(nhom, observed=False).agg(số_stay=("stay_min", "size"),
                                             số_giờ=("stay_min", lambda x: x.sum() / 60),
                                             rad95_m=("rad95_m", "median")).round(1))

408 stay giả, tổng 322 giờ


,số_stay,số_giờ,tốc_độ_trung_vị_kmh,quãng_đường_km,rad95_m,độ_dài_nhãn_giờ,chạm_đầu_cuối_nhãn
mode,,,,,,,
walk,314,254.14,1.98,0.99,107.14,2.48,0.20
bike,33,23.43,3.23,1.00,133.24,2.25,0.30
car,31,22.59,2.78,0.62,106.36,4.08,0.10
bus,16,10.83,3.53,1.09,101.50,1.87,0.25
taxi,6,4.53,2.65,1.38,111.44,2.64,0.33
train,5,2.87,0.00,0.06,5.46,6.84,0.00
run,1,1.91,3.86,2.18,62.44,3.17,0.00
subway,2,1.31,5.57,0.98,130.15,2.13,0.50


,số_stay,số_giờ,rad95_m
med_kmh,,,
< 1 km/h,94,79.9,98.0
1–3 km/h,165,136.2,111.3
3–6 km/h,136,97.8,112.8
> 6 km/h,13,7.7,170.9


## Tiêu chí chốt ngưỡng

1. **Giữ giá trị bài gốc** (D = 200 m, T = 30 phút) nếu tỷ lệ nhận nhầm thấp với các phương tiện xe (bus, car, taxi, train, subway; đề xuất dưới khoảng 5%), và phần lớn stay giả trong nhãn `walk` là đứng yên hoặc đi loanh quanh trong một khu nhỏ (mục 17), tức là không phải lỗi của ngưỡng.
2. **Giảm T** (20 hoặc 10 phút) chỉ khi tỷ lệ nhận nhầm vẫn thấp tương đương, vì T nhỏ giữ thêm các lần ghé ngắn (hữu ích cho POI).
3. **D:** không chọn D làm tỷ lệ nhận nhầm tăng rõ (D lớn gộp đi bộ chậm thành stay). D nhỏ hơn 200 m ít nhận nhầm hơn nhưng dễ làm vỡ stay thật; notebook này không đo được mặt đó, nên giữ 200 m nếu 200 m đã đủ thấp.
4. Ghi rõ giới hạn: chỉ đo nhận nhầm, không đo bỏ sót.

**Quyết định:**
- **D = 200 m, T = 30 phút**, giữ giá trị của Li et al. (2008), mục 5.1.2.
- **Tỷ lệ nhận nhầm ở ngưỡng chọn:** 10,3% thời gian di chuyển có nhãn (69 user, khoảng 4.700 giờ). Phần lớn không phải lỗi của ngưỡng; nhận nhầm khi đang chạy xe thật chỉ khoảng 0,2%.
- **Lý do:**
  - **Tiêu chí 1 đạt:** với xe, tỷ lệ nhận nhầm đều dưới 5% (train 0,9%, bus 2,2%, taxi 2,8%, subway 2,9%, car 3,4%; mục 16).
  - **Stay giả nằm chủ yếu trong nhãn `walk`** (383/483 giờ). Các nhãn này dài (trung vị 2,4 giờ), và trong stay giả 95% điểm nằm trong khoảng 110 m quanh tâm, quãng đường khoảng 1 km: đó là đi loanh quanh trong một khu nhỏ, tức là "ở một nơi" theo nghĩa địa điểm. 129 stay giả có tốc độ trung vị < 1 km/h, tức là đứng yên hẳn dù đang trong nhãn. Chỉ 18 stay giả (10,8 giờ, khoảng 0,2% thời gian có nhãn) có tốc độ > 6 km/h, là nhận nhầm khi đang chạy xe thật (mục 17).
  - **Tiêu chí 2 (T):** T = 30 phút cho ít nhận nhầm nhất trong lưới (10,3% so với 15,1% ở 20 phút và 23,4% ở 10 phút), nên không giảm T.
  - **Tiêu chí 3 (D):** D = 100 m nhận nhầm ít hơn (6,3%), nhưng dễ làm vỡ stay thật: theo EDA mục 13, chỉ 68–78% hai đầu khoảng trống ngắn nằm trong 100 m, so với 75–83% trong 200 m. D = 300 m nhận nhầm nhiều hơn (13,7%). Giữ 200 m.
  - **Tiêu chí 4 (giới hạn):** chỉ đo được nhận nhầm, không đo được bỏ sót. Chỉ 69/182 user có nhãn.